projet ISD2 final: Analyse d'une base de données Formule 1! (2000-2024)

In [2]:
# importation des bibiliothéques nécéssaires au projet 

import pandas as pd 
import numpy as np
import dataframe as df
import os

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# chargement des fichiers de source

Data_Dir = "DataSet_f1"                                 # pointe vers le fichier source du répositoire
Output = "DataSet_F1_Final"                             # nom du fichier complet

NA_VALUES = ["//N", ""]

# chargement de chaque fichier individuellement

results = pd.read_csv(os.path.join(Data_Dir, "results.csv"), na_values = NA_VALUES)
races = pd.read_csv(os.path.join(Data_Dir, "races.csv"), na_values = NA_VALUES)
circuits =  pd.read_csv(os.path.join(Data_Dir, "circuits.csv"), na_values = NA_VALUES)
drivers =  pd.read_csv(os.path.join(Data_Dir, "drivers.csv"), na_values = NA_VALUES)
constructors =  pd.read_csv(os.path.join(Data_Dir, "constructors.csv"), na_values = NA_VALUES)
status =  pd.read_csv(os.path.join(Data_Dir, "status.csv"), na_values = NA_VALUES)
qualifying =  pd.read_csv(os.path.join(Data_Dir, "qualifying.csv"), na_values = NA_VALUES)
pit_stops =  pd.read_csv(os.path.join(Data_Dir, "pit_stops.csv"), na_values = NA_VALUES)
driver_standings =  pd.read_csv(os.path.join(Data_Dir, "driver_standings.csv"), na_values = NA_VALUES)
lap_times =  pd.read_csv(os.path.join(Data_Dir, "lap_times.csv"), na_values = NA_VALUES)

pit_agg = pit_stops.groupby(["raceId", "driverId"]).agg(best_lap_ms = ("milliseconds", "min"), lap_std_ms = ("milliseconds", "std")).reset_index()

drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]

# mise en un seul fichier 

df = results.copy()
df.merge(races[["raceId", "year", "round", "circuitId", "date"]], on = "raceId", how = "left")
df.merge(circuits[["circuitId", "country"]], on = "circuitId", how = "left")
df.merge(drivers[["driverId", "dob"]], on = "driverId", how = "left")
df.merge(constructors[["constructorId", "name"]].rename(collumns = {"name" : "constructor_name"}), on = "constructorId", how = "left")
df.merge(status.rename(collumns = {"status": "status_label"}), on = "statusId", how = "left")
df.merge(pit_agg, on = ["raceId", "driverId"], how = "left")


# on renomme chacun des colonnes(optionel mais pratique)

df = df.rename(columns = {
    "grid":         "grid_position",
    "PositionOrder":     "finish_position",
    "points":       "points_scored",
    "laps":         "laps_completed",
    "status_label": "status",
    "country":      "circuit_country",
    "rank":         "fasted_lap_rank",
})

# choix de nos features -> 12 en total 

Features = [
    "year",                                     
    "round", 
    "grid_position", 
    "finish_position",
    "points_scored",
    "laps_completed",
    "status",
    "constructor_name",
    "circuit_country",
    "driver_age",
    "pit_stop_count",
    "fastest_lap_rank",
]

df = df[features]
df = df[df["year"].between(2000, 2004)].reset_index(drop = true)        # reduit le nombre de lignes: passage de 1950-2024 à 2000-2024

df.to_csv(Output, index = False)